In [7]:
import numpy as np
from scipy.interpolate import interp1d
import h5py

In [8]:
# load ak135 model
model1d = np.loadtxt('ak135.txt')
model1d[:,0] *= 1000  # convert to meters
vp_interp = interp1d(model1d[:,0], model1d[:,1])
vs_interp = interp1d(model1d[:,0], model1d[:,2])
rho_interp = interp1d(model1d[:,0], model1d[:,3])

In [9]:
xmin, xmax = -150000, 150000
ymin, ymax = -150000, 150000
zmin, zmax = -50000, 0
nx, ny, nz = 121, 121, 26
x = np.linspace(xmin, xmax, nx)
y = np.linspace(ymin, ymax, ny)
z = np.linspace(zmin, zmax, nz)
vp1d = vp_interp(z)
vs1d = vs_interp(z)
rho1d = rho_interp(z)

In [10]:
# create 3D initial model
vp_init = np.zeros((nx, ny, nz))
vs_init = np.zeros((nx, ny, nz))
rho_init = np.zeros((nx, ny, nz))
for k in range(nz):
    vp_init[:,:,k] = vp1d[k]
    vs_init[:,:,k] = vs1d[k]
    rho_init[:,:,k] = rho1d[k]

In [11]:
# create ckbd
# start point in x, y, z
x0, y0, z0 = -100000, -100000, -40000
# end point in x, y, z
x1, y1, z1 = 100000, 100000, 0
# number of checkers
npertx, nperty, npertz = 5, 5, 2
# perturbation amplitude
pert_vel = 0.1  # 10%

def _create_tape(xleft, xright, coords):
    dx = coords[1] - coords[0]
    ntaper_left = int((xleft - coords[0]) / dx)
    ntaper_right = int((coords[-1] - xright) / dx)
    return ntaper_left, ntaper_right

ntaper_left, ntaper_right = _create_tape(x0, x1, x)
x_pert = np.zeros_like(x)
x_pert[ntaper_left:x.size-ntaper_right] = \
    np.sin(npertx * np.pi * np.arange(x.size - ntaper_left - ntaper_right) \
    / (x.size - ntaper_left - ntaper_right))

ntaper_left, ntaper_right = _create_tape(y0, y1, y)
y_pert = np.zeros_like(y)
y_pert[ntaper_left:y.size-ntaper_right] = \
    np.sin(nperty * np.pi * np.arange(y.size - ntaper_left - ntaper_right) \
    / (y.size - ntaper_left - ntaper_right))
        
ntaper_left, ntaper_right = _create_tape(z0, z1, z)
z_pert = np.zeros_like(z)
z_pert[ntaper_left:z.size-ntaper_right] = \
    np.sin(npertz * np.pi * np.arange(z.size - ntaper_left - ntaper_right) \
    / (z.size - ntaper_left - ntaper_right))
        
xx, yy, zz = np.meshgrid(x_pert, y_pert, z_pert, indexing='ij')
perturbation = pert_vel * xx * yy * zz

# apply perturbation
vp_ckbd = vp_init * np.exp(perturbation)
vs_ckbd = vs_init * np.exp(perturbation)
rho_ckbd = rho_init * np.exp(perturbation)


In [12]:
# write target model to h5 file
with h5py.File('target_model.h5', 'w') as f:
    f.create_dataset('x', data=x)
    f.create_dataset('y', data=y)
    f.create_dataset('z', data=z)
    f.create_dataset('vp', data=vp_ckbd)
    f.create_dataset('vs', data=vs_ckbd)
    f.create_dataset('rho', data=rho_ckbd)

# write initial model to h5 file
with h5py.File('initial_model.h5', 'w') as f:
    f.create_dataset('x', data=x)
    f.create_dataset('y', data=y)
    f.create_dataset('z', data=z)
    f.create_dataset('vp', data=vp_init)
    f.create_dataset('vs', data=vs_init)
    f.create_dataset('rho', data=rho_init)